# 0. Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from itertools import combinations
from collections import Counter
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [3]:
data_path = '/content/drive/MyDrive/UST/Year 5/Spring/COMP4332/Projects/Project 1/data'

# 1. Helper Functions

## 1.2 Find the most frequent combinations of categories in a df column

In [4]:
def get_top_k_p_combinations(df, comb_p, topk, output_freq=False):
    '''
    params:
        df: input dataframe
        comb_p: number of elements in each combination (e.g., there are two elements in the combination {fried chicken, chicken and waffle}, and three elements in the combination {fried chicken, chicken and waffle, chicken fried rice})
        topk: number of most frequent combinations to retrieve
        output_freq: whether to return the frequencies of retrieved combinations

    return:
        1. output_freq = True: a list X where each element is a tuple containing a combination tuple and corresponding frequency, and the elements are stored in the descending order of their frequencies
        2. output_freq = False: a list X where each element is a tuple containing a combination tuple, and the elements are stored in the descending order of their frequencies
    '''
    def get_category_combinations(categories, comb_p=2):
        return list(combinations(categories, comb_p))
    all_categories_p_combos = df["category"].apply(lambda x: get_category_combinations(x, comb_p)).values.tolist()
    all_categories_p_combos = [tuple(t) for item in all_categories_p_combos for t in item]
    tmp = dict(Counter(all_categories_p_combos))
    sorted_categories_combinations = list(sorted(tmp.items(), key=lambda x: x[1], reverse=True))
    if output_freq:
        return sorted_categories_combinations[:topk]
    else:
        return [t[0] for t in sorted_categories_combinations[:topk]]

## 1.3 Create a OHE (or "wide feature representation") of each row in the column of a df

In [5]:
def get_wide_features(df, selected_categories_to_idx, top_combinations):
    '''
    params:
        df: input dataframe
        selected_categories_to_idx: a dictionary mapping item categories to corrresponding integral indices
        top_combinations: a list containing retrieved mostly frequent combinantions of item categories

    return:
        a numpy array where each row contains the categorical features' binary encodings and cross product transformations for the corresponding row of the input dataframe
    '''
    def categories_to_binary_output(categories):
        binary_output = [0 for _ in range(len(selected_categories_to_idx))]
        for category in selected_categories_to_idx:
            if category in selected_categories_to_idx:
                binary_output[selected_categories_to_idx[category]] = 1
            else:
                binary_output[0] = 1
        return binary_output
    def categories_cross_transformation(categories):
        current_category_set = set(categories)
        corss_transform_output = [0 for _ in range(len(top_combinations))]
        for k, comb_k in enumerate(top_combinations):
            if len(current_category_set & comb_k) == len(comb_k):
                corss_transform_output[k] = 1
            else:
                corss_transform_output[k] = 0
        return corss_transform_output

    category_binary_features = np.array(df.category.apply(lambda x: categories_to_binary_output(x)).values.tolist())
    category_cross_transform_features = np.array(df.category.apply(lambda x: categories_cross_transformation(x)).values.tolist())
    return np.concatenate((category_binary_features, category_cross_transform_features), axis=1)

# 2. Data Handling

## 2.1 Data Exploration

In [6]:
prediction_data = pd.read_csv(data_path + '/prediction.csv')
product_data = pd.read_json(data_path + '/product.json')
review_data = pd.read_csv(data_path + '/review.csv')
validation_data = pd.read_csv(data_path + '/validation.csv')

print(f"Prediction data shape: {prediction_data.shape}")
print(f"Product data shape: {product_data.shape}")
print(f"Review data shape: {review_data.shape}")
print(f"Validation data shape: {validation_data.shape}")

Prediction data shape: (6633, 3)
Product data shape: (6309, 17)
Review data shape: (52512, 5)
Validation data shape: (6596, 3)


In [7]:
prediction_data

,ReviewerID,ProductID,Star
0,A2MK1L1Y74WTWH,B01GT5XDFS,0
1,A19I68RW4PBT29,B00OME9OQQ,0
2,A1UPHTDW5GM12T,B01GSRNLOK,0
3,A1LFIFPYMOJ8RV,B01CUJYMR0,0
4,A10Y597K071WTQ,B004SI455Q,0
...,...,...,...
6628,A23Y4UGTFDMZOP,B00J5327X6,0
6629,A2PFNDDKHOOMZU,B01G0GIXJ2,0
6630,A1K4S4MWXI9E9M,B01FKDKB96,0
6631,AOLHNMI8G8R6K,B00NUDPR66,0


In [8]:
product_data

,category,tech1,description,fit,title,tech2,brand,feature,rank,details,main_cat,similar_item,date,price,imageURL,imageURLHighRes,ProductID
0,"[Kindle Store, Kindle eBooks, Biographies & Me...",,[],,,,Visit Amazon's Frank W. Abagnale Page,[],"59,404 Paid in Kindle Store (","{'File Size:': '1466 KB', 'Print Length:': '30...",Buy a Kindle,,NaT,,[],[],B000FBFMHU
1,"[Kindle Store, Kindle eBooks, Politics & Socia...",,[],,,,Visit Amazon's Karl Marx Page,[],"1,358,073 Paid in Kindle Store (","{'File Size:': '142 KB', 'Print Length:': '160...",Buy a Kindle,,NaT,,[],[],B000FC27TA
2,"[Kindle Store, Kindle eBooks, Romance]",,[],,,,Visit Amazon's Allison Brennan Page,[],"94,006 Paid in Kindle Store (","{'File Size:': '739 KB', 'Print Length:': '416...",Buy a Kindle,,NaT,,[],[],B000FCKPG2
3,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",,[],,,,Visit Amazon's Lynsay Sands Page,[],"31,652 Paid in Kindle Store (","{'File Size:': '1011 KB', 'Print Length:': '38...",Buy a Kindle,,NaT,,[],[],B000GCFWXW
4,"[Kindle Store, Kindle eBooks, Romance]",,[],,,,Visit Amazon's Fern Michaels Page,[],"1,031,468 Paid in Kindle Store (","{'File Size:': '519 KB', 'Print Length:': '320...",Buy a Kindle,,NaT,,[],[],B000JMKRTI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6304,"[Kindle Store, Kindle eBooks, Romance]",,[],,SCARS - Kindle edition,,Visit Amazon's Jaimie Roberts Page,[],"17,812 Paid in Kindle Store (","{'File Size:': '2096 KB', 'Print Length:': '42...",Buy a Kindle,,NaT,,[],[],B01HFFPC2I
6305,"[Kindle Store, Kindle eBooks, Romance]",,[],,Bride Of The Dragon - Kindle edition,,Visit Amazon's Georgette St. Clair Page,[],"220,668 Paid in Kindle Store (","{'File Size:': '4037 KB', 'Print Length:': '17...",Buy a Kindle,,NaT,,[],[],B01HFGNGYI
6306,"[Kindle Store, Kindle eBooks, Literature & Fic...",,[],,Miles (Special Forces Book 3) eBook,,Visit Amazon's KB Winters Page,[],"412,334 Paid in Kindle Store (","{'File Size:': '2444 KB', 'Print Length:': '29...",Buy a Kindle,,NaT,,[],[],B01HFUF1GK
6307,"[Kindle Store, Kindle eBooks, Romance]",,[],,The Billionaire&#39;s Triplets: Book One - Kin...,,Visit Amazon's Mia Caldwell Page,[],"9,672 Free in Kindle Store (","{'File Size:': '5487 KB', 'Print Length:': '27...",Buy a Kindle,,NaT,,[],[],B01HFTVMXM


In [9]:
review_data

,ReviewerID,ProductID,Text,Summary,Star
0,A1XJXYKOWCH9XT,B000FBFMHU,Liked the movie. Loved the book. It really giv...,Liked the movie. Loved the book!,5.0
1,A1K4S4MWXI9E9M,B000FC27TA,Purchased more out of curiosity than any real ...,"Not my favorite, but...",3.0
2,A3LF914GG87TWP,B000FC27TA,"I actually received this text as an ebook, sin...",An interesting read,4.0
3,A1CNQTCRQ35IMM,B000FCKPG2,REVIEWER'S OPINION:\nThis was labeled as roman...,This was labeled romance but there was less ro...,2.0
4,AU510CVD9XDG,B000GCFWXW,I have been saving the Argeneau novels for awh...,Science Fiction not Paranormal Romance,2.0
...,...,...,...,...,...
52507,A3JVZY05VLMYEM,B01FLJUZ0E,She can't do anything right according to her f...,What Can She Do,5.0
52508,A2U06P692IZOSF,B01FLJUZ0E,Better late than never!!\nKitty Konstantine ha...,BART & KITTY CAT MAKE SPARKS FLY!!,5.0
52509,A3RPL8JIS2XMJ3,B01FLJUZ0E,This book was great. Bartholomew finally gets ...,LOVE THE SAINTS,5.0
52510,A1XMFCMIANCQRW,B01FPYJS1M,I read for a honest review for the author.\nTh...,"Loved Lee and Raina together, Ricky is evil an...",4.0


In [10]:
validation_data

,ReviewerID,ProductID,Star
0,A25X28UZCW2J6G,B00K9V6B94,4.0
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0
2,AAUVEEG5YLZAX,B01864DDVO,5.0
3,A3VQLGTYTL5196,B001BXNQ2O,5.0
4,A10JAUCIGVRW9F,B0116MZUS2,5.0
...,...,...,...
6591,A3TC60MGLW1I76,B00EHSUFD8,4.0
6592,AGE0YGLF7L2ZL,B014LQ18CW,4.0
6593,AT2ZB20OCU7X2,B00ZRDPPU0,4.0
6594,A1ACUN6A2LYVMW,B01EKIELGG,1.0


## 2.2 Preprocessing

### 2.2.1 Merging the train_df and val_df

In [11]:
train_df = pd.merge(review_data[['ReviewerID', 'ProductID', 'Star']], product_data, on='ProductID').reset_index(drop=True)
val_df = pd.merge(validation_data, product_data, on='ProductID').reset_index(drop=True)
train_df = train_df[['ReviewerID', 'ProductID', 'Star', 'category', 'brand']]
val_df = val_df[['ReviewerID', 'ProductID', 'Star', 'category', 'brand']]

In [12]:
# brand col --> remove "Visit Amazon's " and " Page"
train_df['brand'] = train_df['brand'].str.replace('Visit Amazon\'s ', '').str.replace(' Page', '')
val_df['brand'] = val_df['brand'].str.replace('Visit Amazon\'s ', '').str.replace(' Page', '')

<ipython-input-12-5e38d88bdda2>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df['brand'] = val_df['brand'].str.replace('Visit Amazon\'s ', '').str.replace(' Page', '')


In [13]:
train_df

,ReviewerID,ProductID,Star,category,brand
0,A1XJXYKOWCH9XT,B000FBFMHU,5.0,"[Kindle Store, Kindle eBooks, Biographies & Me...",Frank W. Abagnale
1,A1K4S4MWXI9E9M,B000FC27TA,3.0,"[Kindle Store, Kindle eBooks, Politics & Socia...",Karl Marx
2,A3LF914GG87TWP,B000FC27TA,4.0,"[Kindle Store, Kindle eBooks, Politics & Socia...",Karl Marx
3,A1CNQTCRQ35IMM,B000FCKPG2,2.0,"[Kindle Store, Kindle eBooks, Romance]",Allison Brennan
4,AU510CVD9XDG,B000GCFWXW,2.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Lynsay Sands
...,...,...,...,...,...
52323,A3JVZY05VLMYEM,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning
52324,A2U06P692IZOSF,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning
52325,A3RPL8JIS2XMJ3,B01FLJUZ0E,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Becca Fanning
52326,A1XMFCMIANCQRW,B01FPYJS1M,4.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Marci Fawn


In [14]:
val_df

,ReviewerID,ProductID,Star,category,brand
0,A25X28UZCW2J6G,B00K9V6B94,4.0,"[Kindle Store, Kindle eBooks, Science Fiction ...",Erin Kellison
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Evangeline Anderson
2,AAUVEEG5YLZAX,B01864DDVO,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Lucia Jordan
3,A3VQLGTYTL5196,B001BXNQ2O,5.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Eve Vaughn
4,A30WJKGX1XG19Q,B00I4KRGPA,2.0,"[Kindle Store, Kindle eBooks, Literature & Fic...",Violet Duke
...,...,...,...,...,...
5437,A3TC60MGLW1I76,B00EHSUFD8,4.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Jaden Skye
5438,AGE0YGLF7L2ZL,B014LQ18CW,4.0,"[Kindle Store, Kindle eBooks, Romance]",Susan Hatler
5439,AT2ZB20OCU7X2,B00ZRDPPU0,4.0,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",Lee Child
5440,A1ACUN6A2LYVMW,B01EKIELGG,1.0,"[Kindle Store, Kindle eBooks, Romance]",Lee Savino


In [15]:
print(train_df.info(), '\n')
print(val_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52328 entries, 0 to 52327
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ReviewerID  52328 non-null  object 
 1   ProductID   52328 non-null  object 
 2   Star        52328 non-null  float64
 3   category    52328 non-null  object 
 4   brand       52328 non-null  object 
dtypes: float64(1), object(4)
memory usage: 2.0+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5442 entries, 0 to 5441
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ReviewerID  5442 non-null   object 
 1   ProductID   5442 non-null   object 
 2   Star        5442 non-null   float64
 3   category    5442 non-null   object 
 4   brand       5442 non-null   object 
dtypes: float64(1), object(4)
memory usage: 212.7+ KB
None


### 2.2.2 Preparing deep categorical features

In [16]:
deep_columns = ['brand']

deep_vocab_lens = []
for col_name in deep_columns:
    # Get unique values of this col
    unique_values = train_df[col_name].unique()

    # Put each unique value into a dict: (value, num), start num range from 1 onwards
    vocab = dict(zip(unique_values, range(1, len(unique_values) + 1)))

    # Add the number of items in the dict (+1 for unknown)
    deep_vocab_lens.append(len(vocab) + 1)

    # Add columns to the train_df to display the idx number of the OHE
    train_df[col_name + '_idx'] = train_df[col_name].apply(lambda x: vocab[x])

# Map ProductID to the _idx columns in train_df
deep_idx_columns = [t + "_idx" for t in deep_columns]
deep_categorical_features = dict(zip(train_df['ProductID'].values, train_df[deep_idx_columns].values.tolist()))

# Convert the _idx columns into np arrays for the model
train_deep_categorical_features = np.array(train_df['ProductID'].apply(lambda x: deep_categorical_features[x]).values.tolist())
val_deep_categorical_features = np.array(val_df['ProductID'].apply(lambda x: deep_categorical_features[x]).values.tolist())

In [17]:
len(train_deep_categorical_features), train_deep_categorical_features

(52328,
 array([[   1],
        [   2],
        [   2],
        ...,
        [3530],
        [3555],
        [3530]]))

In [18]:
len(val_deep_categorical_features), val_deep_categorical_features

(5442,
 array([[2133],
        [  49],
        [ 806],
        ...,
        [ 266],
        [3524],
        [3264]]))

### 2.2.3 Preparing wide features

Preparing binary encoding for each selected category

In [19]:
# Collect all categories from the category column
all_categories = []
for category_list in train_df.category.values:
    # Since category is already a list, we don't need to split it
    for category in category_list:
        all_categories.append(category)

# Sort all unique values of the categories by their frequencies in descending order
from collections import Counter
category_sorted = sorted(Counter(all_categories).items(), key=lambda x: x[1], reverse=True)

# Select top 500 most frequent categories
selected_categories = [t[0] for t in category_sorted[:500]]

# Create a dictionary mapping each selected category to a unique integral index
selected_categories_to_idx = dict(zip(selected_categories, range(1, len(selected_categories) + 1)))

# Map all categories unseen in the df to index 0
selected_categories_to_idx['unk'] = 0

# Create a dictionary mapping each integral index to corresponding category
idx_to_selected_categories = {val: key for key, val in selected_categories_to_idx.items()}

In [20]:
idx_to_selected_categories

{1: 'Kindle Store',
 2: 'Kindle eBooks',
 3: 'Literature & Fiction',
 4: 'Romance',
 5: 'Mystery, Thriller & Suspense',
 6: 'Science Fiction & Fantasy',
 7: 'Religion & Spirituality',
 8: 'Teen & Young Adult',
 9: "Children's eBooks",
 10: 'Health, Fitness & Dieting',
 11: 'Business & Money',
 12: 'Humor & Entertainment',
 13: 'Cookbooks, Food & Wine',
 14: '</span>',
 15: 'Biographies & Memoirs',
 16: 'Kindle Keyboard',
 17: 'Kindle DX',
 18: 'Kindle (2nd Generation)',
 19: 'Kindle (5th Generation)',
 20: 'Activities, Puzzles & Games',
 21: 'Politics & Social Sciences',
 22: 'Arts & Photography',
 23: 'Literature &amp; Fiction',
 24: 'Reference',
 25: 'Kindle Paperwhite',
 26: 'Kindle Paperwhite (5th Generation)',
 27: 'Kindle Touch',
 28: 'Crafts, Hobbies & Home',
 29: 'History',
 30: 'Computers & Technology',
 31: 'Science & Math',
 32: 'Self-Help',
 33: 'Education & Teaching',
 34: 'Word Games',
 35: 'Parenting & Relationships',
 36: 'Comics, Manga & Graphic Novels',
 37: 'Travel',

Preparing cross product transformation for categories

In [21]:
# Get most frequent categories combinantions using the utility function defined previously and store them in the folloing list
top_combinations = []

# Get top 50 most frequent two-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 3, 50, output_freq=False)

# Get top 30 most frequent three-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 3, 30, output_freq=False)

# Get top 20 most frequent four-categories combinantions in the train set
top_combinations += get_top_k_p_combinations(train_df, 4, 20, output_freq=False)

# Convert each combinantion in the list to a set data structure
top_combinations = [set(t) for t in top_combinations]

In [22]:
top_combinations

[{'Kindle Store', 'Kindle eBooks', 'Literature & Fiction'},
 {'Kindle Store', 'Kindle eBooks', 'Romance'},
 {'Kindle Store', 'Kindle eBooks', 'Mystery, Thriller & Suspense'},
 {'Kindle Store', 'Kindle eBooks', 'Science Fiction & Fantasy'},
 {'Kindle Store', 'Kindle eBooks', 'Religion & Spirituality'},
 {'Kindle Store', 'Kindle eBooks', 'Teen & Young Adult'},
 {"Children's eBooks", 'Kindle Store', 'Kindle eBooks'},
 {'Health, Fitness & Dieting', 'Kindle Store', 'Kindle eBooks'},
 {'Business & Money', 'Kindle Store', 'Kindle eBooks'},
 {'Humor & Entertainment', 'Kindle Store', 'Kindle eBooks'},
 {'Cookbooks, Food & Wine', 'Kindle Store', 'Kindle eBooks'},
 {'</span>', 'Kindle Store', 'Kindle eBooks'},
 {'Biographies & Memoirs', 'Kindle Store', 'Kindle eBooks'},
 {'</span>', 'Kindle Store', 'Literature & Fiction'},
 {'</span>', 'Kindle eBooks', 'Literature & Fiction'},
 {'Kindle Keyboard', 'Kindle Store', 'Kindle eBooks'},
 {'Kindle DX', 'Kindle Store', 'Kindle eBooks'},
 {'Kindle (2nd Ge

In [23]:
train_wide_features = get_wide_features(train_df, selected_categories_to_idx, top_combinations)
val_wide_features = get_wide_features(val_df, selected_categories_to_idx, top_combinations)

In [24]:
train_wide_features

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])

In [25]:
val_wide_features

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])

### 2.2.4 Concatenating deep categorical features and wide features as an input list

In [26]:
train_features = []
train_features += [train_deep_categorical_features[:, i] for i in range(train_deep_categorical_features.shape[1])]
train_features.append(train_wide_features)

val_features = []
val_features += [val_deep_categorical_features[:, i] for i in range(val_deep_categorical_features.shape[1])]
val_features.append(val_wide_features)

In [27]:
train_features

[array([   1,    2,    2, ..., 3530, 3555, 3530]),
 array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]])]

In [28]:
val_features

[array([2133,   49,  806, ...,  266, 3524, 3264]),
 array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]])]

# 3. Model Implementation

## 3.1 Set up Dataset and Model

In [29]:
class RatingDataset(Dataset):
    def __init__(self, features, ratings):
        self.deep_categorical = [torch.LongTensor(f) for f in features[:-1]]
        self.wide = torch.FloatTensor(features[-1])
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            *[dc[idx] for dc in self.deep_categorical],
            self.wide[idx],
            self.ratings[idx]
        )

In [30]:
class WideDeepModel(nn.Module):
    def __init__(self, deep_vocab_lens, len_wide, embed_size, hidden_dims=(256, 128, 64), dropout=0.3):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size, embed_size) for vocab_size in deep_vocab_lens
        ])

        # DNN for the deep part
        layers = []
        input_dim = len(deep_vocab_lens) * embed_size
        for hd in hidden_dims:
            layers.append(nn.Linear(input_dim, hd))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hd

        self.dnn = nn.Sequential(*layers)

        # Final FC combining deep + wide
        self.fc = nn.Linear(hidden_dims[-1] + len_wide, 1)

    def forward(self, *args):
        deep_categorical = args[:len(self.embeddings)]
        wide = args[-1]

        embeds = []
        for i, emb_layer in enumerate(self.embeddings):
            embeds.append(emb_layer(deep_categorical[i]))
        embeds = torch.cat(embeds, dim=1)

        dnn_output = self.dnn(embeds)

        combined = torch.cat([dnn_output, wide], dim=1)

        return self.fc(combined).squeeze()

In [31]:
train_ratings = train_df['Star'].values
val_ratings = val_df['Star'].values

## 3.2 Training the model

In [32]:
train_dataset = RatingDataset(train_features, train_ratings)
val_dataset = RatingDataset(val_features, val_ratings)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device", device)

model = WideDeepModel(
    deep_vocab_lens=deep_vocab_lens,
    len_wide=train_wide_features.shape[1],
    embed_size=100
)
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 3
train_losses = []
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for brand, wide, ratings in train_loader:
        optimizer.zero_grad()
        outputs = model(
            brand.to(device),
            wide.to(device)
        )
        loss = criterion(outputs, ratings.to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))
    print(f"Epoch {epoch+1} loss: {train_losses[-1]:.4f}")

using device cuda
Epoch 1 loss: 0.9868
Epoch 2 loss: 0.8524
Epoch 3 loss: 0.8592


## 3.3 Evaluation on validation data

In [33]:
def compute_rmse(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for brand, wide, ratings in loader:
            outputs = model(
                brand.to(device),
                wide.to(device)
            )
            total_loss += nn.MSELoss(reduction='sum')(outputs, ratings.to(device)).item()
    mse = total_loss / len(loader.dataset)
    return torch.sqrt(torch.tensor(mse)).item()

train_rmse = compute_rmse(model, train_loader)
val_rmse = compute_rmse(model, val_loader)

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")

Train RMSE: 0.9217
Validation RMSE: 1.0526
